# pars 

In [1]:
from tmspath_utils import *


In [2]:
date, start_time = import_modules()

detrendType = 'Windowed' #'Windowed'  #'Non-Windowed' #'ICA_Detrend' '

if detrendType == 'Non-Windowed':
    typeOffsetRise, typeOffsetDecay = 'nowind_biexp', 'nowind_biexp'
if detrendType == 'Windowed':
    typeOffsetRise = 'wind_poly_lagrange' #'wind_poly_lagrange' #wind_poly_lagrange' #'wind_singlerise',
    typeOffsetDecay = 'wind_poly_3' #'wind_poly_3' # 'wind_poly_0', 'wind_poly_1', 'wind_poly_2', 'wind_singledecay',
if detrendType == 'ICA_Detrend':
    typeOffsetRise = 'ICA' 
    typeOffsetDecay = 'ICA' 
    
# === Config iniziale ===
json_data = {
    # utils
    'date': date,
    'start_time': start_time,
    'do_filter_and_plot_raw': True,
    'showPlotsEnd': False,
    'r_sfreq': 1024,
    'eeg_type': 'tep',

    # filtering, 
    'l_freq': 0.1, #1.59,
    'h_freq': 45,
    'powerline_freq': 50,
    'broad_band_h_freq': 250,

     # stimulation
    'do_pulseArtifactRej': True,
    'pulse_artifact_rej_timewindow_min' : -0.002, 
    'pulse_artifact_rej_timewindow_max' : 0.008,  
    'emispheric_stimulation': 'SX',
    'seedChans': ['AF3', 'F3', 'Fz', 'FC1'],
    
     # epoching
    'do_prepare_epochs': True,
    'epochs_timewindow_min': -0.1,
    'epochs_timewindow_max': 0.4,
    'baseline_cor_tmin': -0.15,
    'baseline_cor_tmax': 0,
    
     # chans and trials cleaning
    'do_clean_trials_channels': True,
    'do_chan_trials_selection_automatic': True, # if you do True, PARS are in tmspath_utils.py ctrl-f AUTOCHANTRIALSREJ
    'bad_trials': [], # np.arange(0,180).tolist(), #[100], #[13,37,44,51,55], # [] else leave empty
    'bad_channels': [], # ['AF3'], # [] else leave empty
     # nota: nella visualizzazione manuale, i bad trials e bad channels pre-indicated non sono visualizzati 
     # ma sono tolti in automatatico successivamente
    
     # injected artifacts options
    'do_artifact': False, 
    'do_artifact_rise': 0.005,
    'do_artifact_decay': 0.1,
    'do_artifact_gain': -3e-6 * 30,
    'do_artifact_chans': ['Cz'],

     # ica options
    'do_ica': True, # if typeOffsetDecay == typeOffsetRise == 'ICA' make True else no detrend at all
    'do_ica_continuum': False,
    'do_ica_manualCheck': False, 
    'do_ica_eigThresh': 0,
    'do_ica_automaticRej': False if detrendType == 'ICA_Detrend' else True, # è messo in falso nel caso di detrend con ICA per permettere all'operatore di valutare quali sono le componenti offset che IClabel non può conoscere
    'do_label_prob_threshold': 0,

     # detrend options
    'trials_wise': True,  # True: detrend su ogni trial; False: detrend sulla media (TEP)
    'detrendExtremeTechinque': 'max',
    'detrend_type': typeOffsetDecay,
    'do_detrend': typeOffsetDecay != 'ICA', # if FALSE it acts with detrend_overall if True, else no detrend at all
    'do_detrend_onlyOffsetChans': False,
    'detrend_offsetStart': True,
    'detrend_lag_correction': False,
    'detrend_typeOffsetRise': typeOffsetRise,
    'detrend_typeOffsetDecay': typeOffsetDecay,
    'detrend_polOrder_preOffset': 2,
    'detrend_slopeThr': 0.5, # 0.5, important for making the detrend otherwise skip to poly0 without windowing (demeaning)
    'detrend_offsetCorrectionType': 'Gaussian' if 'nowind' not in typeOffsetDecay else False,
    'detrend_offsetOddSamples': 5,
    'detrend_overall': True,
    'detrend_noWindowedOrder': 1, # if detrend_overall True, it uses an overall linear detrend (if 0, is a constant (signal mean), not a line))
    'detrend_maxTimeWindowOffset': 0.020,
    'detrend_minTimeWindowOffset': 0.0,
    'detrend_fitConstraint': False if 'nowind' not in typeOffsetDecay else True,

    # directory and subject
    # RICORDARSI DI METTERE PATH ASSOLUTO!
    'mainDir': r"C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data",
    'sourceData': 'MAYER', #'UNIMI', # MAYER CHALFONT
    'subject': "PP064", #"CE17_18_07_19_000003", #"CE20_07_08_2019_000004" , "CE17_18_07_19_000003" "prova_Mario_0003", #"prova_Betta_0002", #"prova_Mario_0003", #"TEPEMISFEROSXSTIMNETXTIMEASYCUP", #"prova_Betta_0002", #'TEPEMISFEROSXEASYCUP', #"prova_Betta_0002", #"prova_Betta_0001", #"prova_Mario_0003",  #prova_Mario_0003", #"CE20_07_08_2019_000004", #"prova_Mario_0003",
    'dataType': 'ASCII', #'VHDR', #'ASCII', #'VHDR',
    'emispheric_stimulation': 'SX',
    # 'seedChans': ['AF3', 'F3', 'Fz', 'FC1'], #'SX': ['AF3', 'F3', 'Fz', 'FC1'], 'DX': ['Fz', 'AF4', 'F4', 'FC2'],
}

if json_data['emispheric_stimulation'] == 'SX':
    json_data['seedChans'] = ['AF3', 'F3', 'Fz', 'FC1']

elif json_data['emispheric_stimulation'] == 'DX':
    json_data['seedChans'] = ['Fz', 'AF4', 'F4', 'FC2']


main_dir=Path(json_data["mainDir"])
subject=json_data["subject"]
hemisphere=json_data["emispheric_stimulation"]
subject_dir=main_dir/subject
file_stem=subject_dir/f"{subject}EMISFERO{hemisphere}"
print("subject_dir:",subject_dir)
print("subject_dir exists:",subject_dir.exists())
print("fileName:",file_stem)
fileName=str(
    subject_dir
    /f"{subject}EMISFERO{hemisphere}"
)
fit_letter=str(
    json_data["detrend_fitConstraint"]
)[0]
offset_letter=str(
    json_data["detrend_offsetCorrectionType"]
)[0]
extra_note=(
    f"{json_data['detrend_typeOffsetRise']}_"
    f"{json_data['detrend_typeOffsetDecay']}"
)
experiment_dir=subject_dir/f"{date}_{fit_letter}_{offset_letter}_{extra_note}"
json_data["experiment_dir"]=str(experiment_dir)
json_data["analysis_id"]=f"{date}_{subject}"
sub=subject
savePath=str(subject_dir)
print("Input stem:",fileName)
print("Output directory:",experiment_dir)




subject_dir: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064
subject_dir exists: True
fileName: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\PP064EMISFEROSX
Input stem: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\PP064EMISFEROSX
Output directory: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3


# TMS-EEG Processing Pipeline

### 1. Preprocessing & Artifact Removal
1. **Load EEG**
2. **Apply montage**
3. **Detect TMS events**
4. **Remove TMS pulse artifact**
5. **Broad-band filter** (0.1–250 Hz)
6. **Notch filter** (50 / 100 / 150 / 200 / 250 Hz)

---

### 2. Preliminary Cleaning
7. **Create long temporary epochs** (for visual/automatic cleaning)
8. **Automatic / manual trial and channel inspection**
9. **Reject bad trials**
10. **Mark bad channels** in `info["bads"]`  
   Bad channels are **not dropped** at this stage.

---

### 3. Epoching & Detrending
11. **Create final epochs** (-100 to +400 ms) using only retained trials
12. **Preserve all EEG channels**
13. **Propagate bad-channel labels** to final epochs
14. **Downsample** to processing rate, e.g. 1024 Hz
15. **Average reference** using good channels only
16. *Optional:* **Pre-detrend PSD / FOOOF**
17. **Slope and offset analysis**
18. **Selected detrending strategy**

---

### 4. ICA
19. **Prepare ICA copy**
    - bad channels remain in the full data object
    - ICA fitting uses only EEG channels not marked as bad

20. **Fit ICA on good channels only**

21. **Automatic / manual ICA component selection**
    - ICLabel / eigenvalue criteria if enabled
    - manual component inspection if enabled

22. **Apply ICA correction** to the full epoched object

---

### 5. Finalizing
23. **Final filter** (0.1–45 Hz)
24. **Resample** to original sampling rate
25. **Interpolate bad channels**
26. **Final average reference**
27. **Save** `postICA_final` and final plots

# init

In [3]:
json_data, experiment_dir, sub = directorySetup(json_data)


📌 Using provided experiment_dir: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3


In [4]:
experiment_dir

"C:\\Users\\verga\\OneDrive - Scuola Superiore Sant'Anna\\dellXXX-home-Pontedera\\Documenti\\MAIN\\073_tempTMSpath\\TMSpathPipeline\\data\\PP064\\20260721153700_F_G_wind_poly_lagrange_wind_poly_3"

# load_and_prepare_raw_data

In [5]:
raw, events, json_data = load_and_prepare_raw_data(
    json_data=json_data,
    fileName=fileName,
    experiment_dir=experiment_dir,
    sub=sub
)


📌 Using provided experiment_dir: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3
GALNT ASCII CONVERTED FILE"

Patient Data: " "PP064 PP064" "19/07/2010"

Trace date: " "06/07/2026"

Start second: " 0

Seconds: " 303

Sampling rate : " 4096

Values are expressed in [uV]"

TR 00", "TR 01", "TR 02", "TR 03", "TR 04", "TR 05", "TR 06", "TR 07", "TR 08", "TR 09", "TR 10", "TR 11", "TR 12", "TR 13", "TR 14", "TR 15", "TR 16", "TR 17", "TR 18", "TR 19", "TR 20", "TR 21", "TR 22", "TR 23", "TR 24", "TR 25", "TR 26", "TR 27", "TR 28", "TR 29", "TR 30", "TR 31", "TR 32"

['AF3', 'FC1', 'FC2', 'AF4', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'T3', 'C3', 'Cz', 'C4', 'T4', 'FC6', 'T5', 'P3', 'Pz', 'P4', 'T6', 'PO3', 'TP9', 'Oz', 'TP10', 'PO4', 'FPz', 'CP1', 'CP2', 'CP5', 'CP6', 'TM', 'MK']
Il canale MK è presente nel DataFrame.
Il canale EMG non è presente nel DataFrame

# computeBasicSteps

In [6]:
raw_cleaned, epochs, detrendedEpochs, temp_epochs, json_data = computeBasicSteps(
    raw, events, json_data, experiment_dir, sub, computeFOOOF=False
)



🔧 [PP064] Step 1: Rimozione artefatto TMS + PSD
📉 Plotting PSD before pulse removal...
NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 0.500 (s)
Plotting power spectral density (dB=True).
⚡ Removing TMS pulse artifact...
📉 Plotting PSD after pulse removal...
NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 0.500 (s)
Plotting power spectral density (dB=True).
🔧 [PP064] Step 2: Filtro broad-band e notch
DO-BROADBAND------------------------------------
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 2.5e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 12 (effective, after forward-backward)
- Cutoffs at 0.10, 250.00 Hz: -6.02, -6.02 dB

NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 0.50

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    2.1s


NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 0.500 (s)
Plotting power spectral density (dB=True).
[INFO] Raw filtered and saved → C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3\6.pkls\PP064_raw.pkl
🔧 [PP064] Step 3: Pulizia epoche e canali artefattati
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 45 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 12 (effective, after forward-backward)
- Cutoffs at 0.10, 45.00 Hz: -6.02, -6.02 dB

Not setting metadata
160 matching events found
Setting baseline interval to [-0.800048828125, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 160 events and 36046 original

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:41<00:00,  1.33s/it]


📉 [PP064] Step 2: Calcolo delle pendenze (computeSlopes_v4)


...

Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 1.00 Hz
- Upper transition bandwidth: 1.00 Hz
- Filter length: 3381 samples (3.302 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.0s


Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 1.00 Hz
- Upper transition bandwidth: 1.00 Hz
- Filter length: 3381 samples (3.302 s)



[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s


NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 0.500 (s)
Plotting power spectral density (dB=True).
Averaging across epochs...
✅ [PP064] Detrending completato.


🔍 [PP064] Step 3: Compute FOOOF


In [ ]:
from pathlib import Path
import numpy as np

tep_fe_dir = Path(experiment_dir) / "5.Extra" / "FE"
tep_fe_dir.mkdir(parents=True, exist_ok=True)

tep_event_bundle_path = tep_fe_dir / f"{sub}_TEP_event_bundle.npz"

np.savez(
    tep_event_bundle_path,
    events=np.asarray(epochs.events, dtype=int),
    sfreq=float(epochs.info["sfreq"]),
    tmin=float(epochs.tmin),
    tmax=float(epochs.tmax),
    baseline=np.asarray(epochs.baseline if epochs.baseline is not None else [np.nan, np.nan], dtype=float),
    first_samp=int(getattr(epochs, "first_samp", 0))
)

json_data["TEP_event_bundle_path"] = str(tep_event_bundle_path)
print("TEP event bundle saved:", tep_event_bundle_path)


# ICAprocessing

In [7]:
if json_data['do_ica']:
    dataToICA = detrendedEpochs if json_data['detrend_overall'] else epochs
    postICA_final, json_data = ICAprocessing(
        dataToICA,
        json_data, experiment_dir, sub,
        autoReject=json_data['do_ica_automaticRej'],
        manualCheck=json_data['do_ica_manualCheck'],
        computeFOOOF=False
    )
    
    do_run_gif=True
    if do_run_gif:
        from pathlib import Path
        postica_subpath = Path("4.postICA") / json_data["ICA_timestamp"]   
        gif_path, json_data = create_butterfly_topomap_gif(
            epochs=postICA_final,
            json_data=json_data,
            experiment_dir=experiment_dir,
            sub=sub,
            saveNote="postICA_final",
            subPath=str(postica_subpath),
            tmin=-0.1,
            tmax=0.45,
            step=0.002,
            xlim=(-0.1, 0.45),
            vlim=(-5, 5),   # Volt
            sphere=0.095,
            duration=50,
            transparency=False,
            save_static=True
        )
        print(gif_path)
        
    






⚙️ ICA eigThresh = 0, label_prob_threshold = 0
[INFO] Oggetto passato direttamente
📌 Bad channels marked before ICA: ['CP2', 'Fpz', 'T3']
🧠 ICA fit on good channels only: 28 channels
🧠 ICA n_components: 27
Fitting ICA to data using 28 channels (please be patient, this may take a while)
Selecting by number: 27 components
Fitting ICA took 6.7s.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✅ IC 0: brain (prob: 0.99) – mantenuto automaticamente
✅ IC 1: brain (prob: 0.97) – mantenuto automaticamente
✅ IC 2: brain (prob: 0.99) – mantenuto automaticamente
✅ IC 3: brain (prob: 0.96) – mantenuto automaticamente
✅ IC 4: brain (prob: 0.96) – mantenuto automaticamente
❌ IC 5: other (prob: 0.58) – escluso automaticamente
✅ IC 6: brain (prob: 0.98) – mantenuto automaticamente
✅ IC 7: brain (prob: 1.00) – mantenuto automaticamente
✅ IC 8: brain (prob: 0.96) – mantenuto automaticamente
❌ IC 9: muscle artifact (prob: 0.51) – escluso automaticamente
❌ IC 10: muscle art

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.9s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    1.3s
[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:    1.9s
[Parallel(n_jobs=1)]: Done 881 tasks      | elapsed:    2.5s
[Parallel(n_jobs=1)]: Done 1151 tasks      | elapsed:    3.6s
[Parallel(n_jobs=1)]: Done 1457 tasks      | elapsed:    4.5s
[Parallel(n_jobs=1)]: Done 1799 tasks      | elapsed:    5.7s
[Parallel(n_jobs=1)]: Done 2177 tasks      | elapsed:    6.7s
[Parallel(n_jobs=1)]: Done 2591 tasks      | elapsed:    7.8s
[Parallel(n_jobs=1)]: Done 3041 tasks      | elapsed:    9.2s
[Parallel(n_jobs=1)]: Done 3527 tasks      | elapsed:   10.4s
[Parallel(n_jobs=1)]: Done 4049 tasks      | elapsed:   11.6s


🧩 Interpolating bad channels after ICA: ['CP2', 'Fpz', 'T3']
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 86.5 mm
Computing interpolation matrix from 28 sensor positions
Interpolating 3 sensors
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 0.500 (s)
Plotting power spectral density (dB=True).
Averaging across epochs...


2026-07-21 15:44:28,142 - matplotlib.animation - WARNING - MovieWriter ffmpeg unavailable; using Pillow instead.
2026-07-21 15:44:28,158 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.PillowWriter'>


Video salvato in: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3\4.postICA\20260721154336/PP064_scalpmaptime_postICA.gif
✅ GIF salvata: C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3\4.postICA\20260721154336\PP064_postICA_final_butterfly_topomap_opaque.gif
C:\Users\verga\OneDrive - Scuola Superiore Sant'Anna\dellXXX-home-Pontedera\Documenti\MAIN\073_tempTMSpath\TMSpathPipeline\data\PP064\20260721153700_F_G_wind_poly_lagrange_wind_poly_3\4.postICA\20260721154336\PP064_postICA_final_butterfly_topomap_opaque.gif


# save and load

In [8]:
json_data = saveLoadTestFinal(postICA_final, json_data, experiment_dir, sub, start_time)

QFileDialog
make_standard_montage
label_components
PrepPipeline
NoisyChannels
datetime
euclidean
Path
interp1d
resample
expon
gamma
laplace
linregress
norm
poisson
rayleigh
t
uniform
ARIMA
FuncAnimation
tabulate
tqdm
notebook_tqdm
create_info
EvokedArray
RawArray
FOOOF
gen_power_spectrum
set_random_seed
plot_spectra
plot_annotated_model
import_modules
make_json_serializable
directorySetup
directorySetup_old_20260416
loadEDF
loadASCII
computeDetrendSteps
add_exp_artifact
computeBasicSteps
load_and_prepare_raw_data
run_ica_continuum_pipeline
remove_tms_artifact_and_plot_psd
filter_and_plot_raw
clean_trials_channels
clean_trials_channels_20072026
clean_trials_channels_old_20260416
clean_trials_channels_20260415
apply_notch_to_offsetChans
run_detrend_pipeline
notch_filter_offset_chans
apply_notch_filter
prepare_epochs
analyze_offset_times
check_detrend_need
add_TEP_to_json
ICAprocessing
ICAprocessing_old_20260416
computeFeatExtraction
computeFeatExtraction_v2
plot_TEP_1d_with_shading
saveL